# Homework 6 Template

This homework keeps the same **three-part architecture** as the lecture notebook:

1. obtain market data and compute basic statistics  
2. visualize market behavior  
3. build a Markowitz mean-variance portfolio  

Use **Yahoo Finance** through the `yfinance` package and keep the sample period fixed at **2022-01-01** to **2025-01-01**.


## Requirements and notes

- Use the lecture watchlist: `SPY`, `VEA`, `EEM`, `AGG`.
- Use `TRADING_DAYS = 252` and `RISK_FREE_RATE = 0.02`.
- Keep the portfolio **long-only** and require the weights to sum to 1.
- Export at least **three figures** and **three tables** to the output folder.
- Because Yahoo Finance is a live online source, rerun the notebook once before submission to make sure the download succeeds.
- If you open the notebook outside the repository in Colab, you may need to set `PROJECT_ROOT` manually.


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "outputs").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# If needed in Colab, you can manually override the root like this:
# PROJECT_ROOT = Path("/content/big-data-analysis-course-english")

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "homework_06_student"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR


**Optional setup**


In [ ]:
%pip install -q yfinance scipy seaborn ipywidgets


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import scipy.optimize as sco

warnings.filterwarnings("ignore")

TRADING_DAYS = 252
RISK_FREE_RATE = 0.02
START_DATE = "2022-01-01"
END_DATE = "2025-01-01"
SIMULATION_SIZE = 5000

def normalize_datetime_column(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df = df.copy()
    df[col] = pd.to_datetime(df[col])
    try:
        df[col] = df[col].dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass
    return df

def get_price_block(downloaded: pd.DataFrame, field: str) -> pd.DataFrame:
    if isinstance(downloaded.columns, pd.MultiIndex):
        block = downloaded.xs(field, axis=1, level=0)
    else:
        block = downloaded[[field]].copy()
    block.columns = [str(c) for c in block.columns]
    return block

def wide_to_long(price_df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    out = price_df.stack(dropna=False).rename(value_name).reset_index()
    out.columns = ["date", "ticker", value_name]
    return out


# Part 1. Obtain market data and compute basic statistics


In [ ]:
institutional_universe = pd.DataFrame(
    {
        "ticker": ["SPY", "VEA", "EEM", "AGG"],
        "instrument_name": [
            "SPDR S&P 500 ETF Trust",
            "Vanguard FTSE Developed Markets ETF",
            "iShares MSCI Emerging Markets ETF",
            "iShares Core U.S. Aggregate Bond ETF",
        ],
        "asset_class": ["Equity", "Equity", "Equity", "Bond"],
        "market_focus": [
            "United States",
            "Developed Markets ex U.S.",
            "Emerging Markets",
            "U.S. Investment-Grade Bonds",
        ],
    }
)

ticker_list = institutional_universe["ticker"].tolist()
instrument_name_map = dict(zip(institutional_universe["ticker"], institutional_universe["instrument_name"]))

recent_download = yf.download(
    tickers=ticker_list,
    period="5d",
    interval="1d",
    auto_adjust=True,
    progress=False,
    threads=False,
    group_by="column",
)

recent_close = get_price_block(recent_download, "Close")
latest_snapshot = recent_close.ffill().iloc[-1].rename("latest_close").to_frame().reset_index()
latest_snapshot.columns = ["ticker", "latest_close"]
latest_snapshot = latest_snapshot.merge(institutional_universe, on="ticker", how="left")
latest_snapshot.to_csv(MODULE_OUTPUT_DIR / "latest_snapshot.csv", index=False)

multi_download = yf.download(
    tickers=ticker_list,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False,
    threads=False,
    group_by="column",
)

close_prices = get_price_block(multi_download, "Close")
open_prices = get_price_block(multi_download, "Open")

close_long = wide_to_long(close_prices, "close")
open_long = wide_to_long(open_prices, "open")

market_df = (
    close_long
    .merge(open_long, on=["date", "ticker"], how="left")
    .dropna(subset=["close"])
)

market_df = normalize_datetime_column(market_df, "date")
market_df["daily_return"] = market_df.groupby("ticker")["close"].pct_change()
market_df = market_df.merge(institutional_universe, on="ticker", how="left")

latest_snapshot


## Task 1. Annualized return, annualized volatility, and Sharpe ratio

Complete the table below. Then sort the assets by **Sharpe ratio** and export the final table to CSV.


In [ ]:
asset_metrics = (
    market_df
    .dropna(subset=["daily_return"])
    .groupby(["ticker", "instrument_name", "asset_class"])["daily_return"]
    .agg(["mean", "std"])
    .reset_index()
)

# TODO: finish the formulas below.
# asset_metrics["annual_return"] = ...
# asset_metrics["annual_volatility"] = ...
# asset_metrics["sharpe_ratio"] = ...

# TODO: keep a clean final table and export it.
# asset_metrics = asset_metrics.sort_values("sharpe_ratio", ascending=False)
# asset_metrics.to_csv(MODULE_OUTPUT_DIR / "asset_metrics.csv", index=False)

asset_metrics.head()


## Task 2. Grouped descriptive statistics

Create a grouped descriptive-statistics table for `daily_return` and `close`, rounded to four decimals.


In [ ]:
# TODO: create a grouped descriptive-statistics table and export it.
# Suggested structure:
# summary_table = (
#     market_df.groupby("instrument_name")[["daily_return", "close"]]
#     .describe()
#     .round(4)
#     .T
# )
# summary_table.to_csv(MODULE_OUTPUT_DIR / "grouped_summary_table.csv")
# summary_table


# Part 2. Visualize market data


## Task 3. Opening and closing prices for SPY

Create a line chart for SPY opening and closing prices.  
Include a title, x-axis label, y-axis label, and legend.  
Save the figure as `homework6_spy_open_close.png`.


In [ ]:
spy = market_df[market_df["ticker"] == "SPY"].copy()

# TODO: create the figure here.
# Suggested idea:
# plt.figure(figsize=(15, 5))
# plt.plot(...)
# plt.savefig(MODULE_OUTPUT_DIR / "homework6_spy_open_close.png", bbox_inches="tight")
# plt.show()

spy.head()


## Task 4. Multi-asset daily-return line plot

Create a multi-asset line plot of daily returns for all four ETFs and save it as `homework6_daily_return_lineplot.png`.


In [ ]:
# TODO: create a line plot using seaborn or matplotlib.
# Suggested columns: x="date", y="daily_return", hue="instrument_name"
# Save the figure to MODULE_OUTPUT_DIR.


## Task 5. Compounded cumulative return plot

Compute **compounded cumulative returns** and create a line plot named `homework6_cumulative_return.png`.


In [ ]:
market_df = market_df.sort_values(["ticker", "date"]).copy()

# TODO: compute compounded cumulative returns.
# market_df["cumulative_return"] = (
#     market_df.groupby("ticker")["daily_return"]
#     .transform(lambda s: ...)
# )

# TODO: draw the line plot and save it.


# Part 3. Markowitz mean-variance portfolio


In [ ]:
returns = (
    market_df[["ticker", "daily_return", "date"]]
    .pivot_table(values="daily_return", index="date", columns="ticker")
    .dropna(how="all")
    .dropna()
)

noa = len(ticker_list)
bounds = ((0, 1),) * noa
constraints = {"type": "eq", "fun": lambda x: np.sum(x) - 1}

def statistics(weights):
    weights = np.array(weights)
    port_return = np.sum(returns.mean() * TRADING_DAYS * weights)
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * TRADING_DAYS, weights)))
    sharpe = (port_return - RISK_FREE_RATE) / port_volatility
    return np.array([port_return, port_volatility, sharpe])

def negative_sharpe(weights):
    return -statistics(weights)[2]

def portfolio_vol(weights):
    return statistics(weights)[1]

returns.head()


## Task 6. Monte Carlo portfolio simulation

Simulate `SIMULATION_SIZE` random portfolios.  
Store expected return, expected volatility, Sharpe ratio, and portfolio weights.  
Then create a scatter plot named `homework6_monte_carlo_portfolios.png`.


In [ ]:
np.random.seed(42)

# TODO: initialize empty containers:
# portfolio_returns = []
# portfolio_volatility = []
# portfolio_sharpe = []
# portfolio_weights = []

# TODO: loop over SIMULATION_SIZE random portfolios.
# Each portfolio should be long-only and fully invested.

# TODO: convert the results to arrays or a DataFrame.
# TODO: create the scatter plot and save it.


## Task 7. Maximum-Sharpe and minimum-volatility portfolios

Use `scipy.optimize.minimize` with the `SLSQP` method to compute:

1. the maximum-Sharpe portfolio  
2. the minimum-volatility portfolio  

Export the two weight tables to:
- `max_sharpe_weights.csv`
- `min_vol_weights.csv`


In [ ]:
# TODO: compute the maximum-Sharpe portfolio with SLSQP.
# Suggested start:
# opt_sharpe = sco.minimize(
#     negative_sharpe,
#     noa * [1.0 / noa],
#     method="SLSQP",
#     bounds=bounds,
#     constraints=constraints,
# )

# TODO: compute the minimum-volatility portfolio with SLSQP.
# TODO: create and export the two weight tables.


## Task 8. Efficient frontier

Create a sequence of target returns, minimize portfolio volatility for each target return, and draw the efficient frontier.  
Save the final figure as `homework6_efficient_frontier.png`.


In [ ]:
# TODO: create a sequence such as:
# target_returns = np.linspace(
#     (returns.mean() * TRADING_DAYS).min(),
#     (returns.mean() * TRADING_DAYS).max(),
#     100,
# )

# TODO: for each target return, minimize portfolio volatility.
# TODO: store the frontier points and export them if you want.
# TODO: draw the final figure with:
# - Monte Carlo points
# - efficient frontier
# - maximum-Sharpe portfolio
# - minimum-volatility portfolio


## Task 9. Short written interpretation

Write **3–5 English sentences** that answer the questions below:

1. Which ETF has the highest sample Sharpe ratio in your run?
2. Which portfolio is more conservative: the maximum-Sharpe portfolio or the minimum-volatility portfolio?
3. What does the efficient frontier show about diversification?


In [ ]:
# Write your short interpretation here.
